# Subsurface NEB Calculation

Script-generator + analysis notebook for H permeation NEB (Hop A and Hop B).

**Hop A** — H* surface site → H subsurface-1 oct site (between layers 11–12)  
**Hop B** — H subsurface-1 oct → H subsurface-2 oct (between layers 10–11), one-to-one with Hop A

---

**Workflow**

1. Build subsurface graph from the relaxed slab.  
2. `orchestrate_hopa_neb(dry_run=True)` → writes FS-min + NEB scripts, no submission.  
3. *(cluster)* Submit `hopa_fsmin_array.sh` (GPU).  
4. `orchestrate_hopb_neb(dry_run=True)` → Hop B scripts (IS = Hop A FS relaxed).  
5. *(cluster)* Submit `hopa_neb_array.sh` + `hopb_fsmin_array.sh` in parallel.  
6. *(cluster)* Submit `hopb_neb_array.sh`.  
7. `orchestrate_vibrations()` → IS + TS frequency scripts for both hops.  
8. Analysis cells: load barriers, plot MEPs.


## Cell 1 — Imports & configuration

Edit all-caps variables below before running.

In [ ]:
import os, sys
import json

parent_dir = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from models.config import (
    MACE_MODEL_ASE, BASE_DIR,
    E2T_7, MASSES_7, ELEM_STR_7,
    SLURM_DEFAULTS, Z_FREEZE_CUTOFF,
    N_REPLICAS, SPRING_CONST, NEB_FTOL,
)

# ── Paths ─────────────────────────────────────────────────────────────────────
WORK_DIR            = os.path.join(BASE_DIR, 'calculation')
RELAXED_SLAB_PATH   = os.path.join(WORK_DIR, 'slabs', 'slab_relaxed.lammps')
SURFACE_SITES_JSON  = os.path.join(WORK_DIR, 'slabs', 'surface_sites.json')

# Phase B dedup IS labels: list of (sid, h_atom_{sid}_relaxed.lammps) tuples
# Replace with the actual dedup output from neb_calculation.ipynb Section B.
PHASE2_H_DIR        = os.path.join(WORK_DIR, 'adsorption', 'h_atom')

# Output directory for subsurface NEB jobs
SUB_NEB_DIR         = os.path.join(WORK_DIR, 'neb_subsurface')
VIB_DIR             = os.path.join(WORK_DIR, 'vibrations')

# ── NEB parameters ────────────────────────────────────────────────────────────
N_IMAGES     = N_REPLICAS   # 18
SPRING_K     = SPRING_CONST # 1.0 eV/Å²
NEB_FTOL_VAL = NEB_FTOL     # 0.05 eV/Å

# ── SLURM configs ─────────────────────────────────────────────────────────────
GPU_SLURM = dict(SLURM_DEFAULTS, partition='multigpu', time='04:00:00')
NEB_SLURM = dict(SLURM_DEFAULTS, partition='short',
                 gpu=None, cpus_per_task=16, time='12:00:00')
VIB_SLURM = dict(SLURM_DEFAULTS, partition='short',
                 gpu=None, cpus_per_task=8, time='06:00:00')

print('Config loaded.')
print(f'  RELAXED_SLAB_PATH : {RELAXED_SLAB_PATH}')
print(f'  SURFACE_SITES_JSON: {SURFACE_SITES_JSON}')
print(f'  SUB_NEB_DIR       : {SUB_NEB_DIR}')
print(f'  VIB_DIR           : {VIB_DIR}')

## Cell 2 — Build subsurface graph

Voronoi-classifies all interstitial oct sites in the slab and connects them to surface sites.

In [ ]:
from models.subsurface_graph import build_subsurface_graph, connect_to_surface

G, subsurface_sites = build_subsurface_graph(
    slab_path=RELAXED_SLAB_PATH,
    surface_sites_json_path=SURFACE_SITES_JSON,
)

# Load surface sites for connect_to_surface
with open(SURFACE_SITES_JSON) as f:
    surface_sites_data = json.load(f)

# Read cell from slab
from ase.io import read as ase_read
slab_atoms = ase_read(RELAXED_SLAB_PATH, format='lammps-data', atom_style='atomic')
cell = slab_atoms.get_cell()

surface_connections = connect_to_surface(subsurface_sites, surface_sites_data, cell)

sub1 = [s for s in subsurface_sites if s.get('layer_classification') == 'subsurface_1']
sub2 = [s for s in subsurface_sites if s.get('layer_classification') == 'subsurface_2']
print(f'Subsurface-1 sites : {len(sub1)}')
print(f'Subsurface-2 sites : {len(sub2)}')
print(f'Surface connections: {len(surface_connections)}')

## Cell 3 — Collect dedup IS labels from Phase B

Reads the unique IS labels produced by Section B of `neb_calculation.ipynb`.

In [ ]:
import glob

# Each dedup IS: a relaxed h_atom_{sid}_relaxed.lammps file.
# Adjust glob pattern to match your directory layout.
dedup_is_labels = [
    (os.path.basename(p).replace('h_atom_', '').replace('_relaxed.lammps', ''),
     p)
    for p in sorted(glob.glob(os.path.join(PHASE2_H_DIR, 'h_atom_*_relaxed.lammps')))
]
print(f'Found {len(dedup_is_labels)} dedup IS structures.')
for sid, path in dedup_is_labels[:5]:
    print(f'  sid={sid!r}  path={path}')

## Cell 4 — Orchestrate Hop A NEB

Writes FS-min scripts + NEB scripts for each unique IS site.  
Set `dry_run=False` to submit immediately.

In [ ]:
from models.neb_subsurface import orchestrate_hopa_neb

hopa_out = orchestrate_hopa_neb(
    dedup_is_labels   = dedup_is_labels,
    subsurface_graph  = (G, subsurface_sites),
    surface_connections = surface_connections,
    outdir            = os.path.join(SUB_NEB_DIR, 'hopa'),
    masses            = MASSES_7,
    e2t               = E2T_7,
    slurm_opts        = GPU_SLURM,
    neb_slurm_opts    = NEB_SLURM,
    n_images          = N_IMAGES,
    spring_const      = SPRING_K,
    neb_ftol          = NEB_FTOL_VAL,
    dry_run           = True,   # ← set False to submit
)

hopa_jobs = hopa_out['jobs']
print(f"Hop A: {hopa_out['n_jobs']} jobs  status={hopa_out['status']}")
print(f"  FS-min array : {hopa_out['fsmin_array']}")
print(f"  NEB array    : {hopa_out['neb_array']}")

## Cell 5 — Orchestrate Hop B NEB

Run **after Hop A FS-min jobs complete** on the cluster.  
IS for Hop B = relaxed sub1 structure from Hop A.

In [ ]:
from models.neb_subsurface import orchestrate_hopb_neb

hopb_out = orchestrate_hopb_neb(
    hopa_jobs        = hopa_jobs,
    hopa_outdir      = os.path.join(SUB_NEB_DIR, 'hopa'),
    subsurface_graph = (G, subsurface_sites),
    outdir           = os.path.join(SUB_NEB_DIR, 'hopb'),
    masses           = MASSES_7,
    e2t              = E2T_7,
    slurm_opts       = GPU_SLURM,
    neb_slurm_opts   = NEB_SLURM,
    n_images         = N_IMAGES,
    spring_const     = SPRING_K,
    neb_ftol         = NEB_FTOL_VAL,
    dry_run          = True,   # ← set False to submit
)

hopb_jobs = hopb_out['jobs']
print(f"Hop B: {hopb_out['n_jobs']} jobs  status={hopb_out['status']}")
print(f"  FS-min array : {hopb_out['fsmin_array']}")
print(f"  NEB array    : {hopb_out['neb_array']}")

## Cell 6 — Orchestrate vibrational frequency calculations

Run **after Hop A and Hop B NEB jobs have converged**.  
Computes IS + TS partial-Hessian frequencies (H + 6 nearest metals) for both hops.

In [ ]:
from models.vibrations import collect_is_ts_paths, orchestrate_vibrations

# Collect IS + TS paths for both hops
pairs_a = collect_is_ts_paths(hopa_jobs, hop='hopa', is_key='is_path')
pairs_b = collect_is_ts_paths(hopb_jobs, hop='hopb', is_key='hopb_is')
all_pairs = pairs_a + pairs_b

print(f'Structures for vibration: {len(all_pairs)}')
for lbl, path in all_pairs[:4]:
    print(f'  {lbl}: {path}')

vib_out = orchestrate_vibrations(
    structure_paths = all_pairs,
    outdir          = VIB_DIR,
    mace_model_path = MACE_MODEL_ASE,
    slurm_opts      = VIB_SLURM,
    delta           = 0.01,
    device          = 'cpu',
    dry_run         = True,   # ← set False to submit
)

print(f"\nVibration jobs: {len(vib_out)}  (IS + TS for each hop)")

---
## Analysis — run after all cluster jobs complete

### 7a. Load Hop A and Hop B barriers

In [ ]:
import pandas as pd
from models.parsers import parse_barrier_file

def load_hop_barriers(jobs, hop_label):
    rows = []
    for job in jobs:
        bf = job.get('barrier_file', '')
        if not os.path.exists(bf):
            continue
        d = parse_barrier_file(bf)
        d['sid']  = job['sid']
        d['hop']  = hop_label
        rows.append(d)
    return pd.DataFrame(rows)

df_a = load_hop_barriers(hopa_jobs, 'hopa')
df_b = load_hop_barriers(hopb_jobs, 'hopb')
df   = pd.concat([df_a, df_b], ignore_index=True)

print(df[['hop', 'sid', 'E_abs', 'E_des', 'delta_E', 'converged']].to_string(index=False))

### 7b. Plot MEPs

In [ ]:
import matplotlib.pyplot as plt
from models.parsers import parse_neb_path

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)

for ax, (jobs, label, color) in zip(axes, [
    (hopa_jobs, 'Hop A  (surface → sub1)', 'steelblue'),
    (hopb_jobs, 'Hop B  (sub1 → sub2)',    'coral'),
]):
    for job in jobs:
        pf = job.get('path_file', '')
        if not os.path.exists(pf):
            continue
        frac, _, dE = parse_neb_path(pf)
        ax.plot(frac, dE, color=color, alpha=0.5, lw=1.2)
    ax.set_xlabel('Reaction coordinate')
    ax.set_ylabel('ΔE  [eV]')
    ax.set_title(label)
    ax.axhline(0, color='k', lw=0.8, ls='--')

plt.tight_layout()
plt.savefig(os.path.join(SUB_NEB_DIR, 'mep_overlay.png'), dpi=150)
plt.show()